# DeepNet forecast for IMD

emile esmaili

In [ ]:
import utils.dataloader as dataloader
import utils.plots as plots
import utils.training as training
import utils.preprocessing as preprocessing
import os
import numpy as np
import xarray as xr
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

##  User inputs

In [ ]:
obs = "IMD"
model = "GEFS"
domain = [67, 98, 7, 38] # west east south north. for Unet's check that lat and lot make a square divisible by 8, ie 24x24, 32x32, 64x64
season = "May-Sep"
n_bootstraps = 10
years = (1989, 2018)

In [ ]:
download = True
if download:
    os.makedirs(f'download/{model}_{obs}', exist_ok=True)
os.makedirs(f'models/{model}_{obs}', exist_ok=True)
os.makedirs(f'figures/{model}_{obs}', exist_ok=True)
os.makedirs('outputs',exist_ok=True)

In [ ]:
week = "wk3-4"  #wk1, wk2 or wk3-4

## Load data

In [ ]:
x, y = dataloader.get_data(years=years, download = download,week=week,obs=obs, domain=domain, season=season, 
                           model=model,regrid=None)


## Baseline: Extended Logistic Regression

In [ ]:
xtrain_list, ytrain_list, xval_list, yval_list = preprocessing.bootstrap_splits_ELR(x, y, n_bootstraps= n_bootstraps)

In [ ]:
rpss_train_list_elr, rpss_test_list_elr, predictions_list_elr, y_test_oh_list_elr = training.train_elr(xtrain_list, ytrain_list, xval_list, yval_list)

In [ ]:
#levels=[-0.3,-0.2,-0.1,-0.05, 0, 0.05,0.1, 0.2,0.4]
plots.plot_rpss_elr(rpss_train_list_elr, rpss_test_list_elr, week=week, obs=obs,
                    levels=None)

## DeepNet: Unet

In [ ]:
xtrain_list, ytrain_list, xval_list, yval_list, xtest_list, ytest_list = preprocessing.bootstrap_splits(x, y, n_bootstraps=n_bootstraps)

In [ ]:
#print train val test years for each bootstrap
for i in range(n_bootstraps):
    print('Bootstrap', i+1)
    print('Train years:', set(xtrain_list[i]['T'].dt.year.values))
    print('Validation years:', set(xval_list[i]['T'].dt.year.values))
    print('Test years:', set(xtest_list[i]['T'].dt.year.values))
    print('')

In [ ]:
architecture = "unet"   #unet or cnn or mlp
#for unet you can specifiy the architecture parameters if training
architecture_params = {"n_blocks": 3, "filters": 2, "ct_kernel": (3,3)} #if unet
# you can also tune the architecture parameters, takes very long
tuning_grid = {"n_blocks": [3,4,5], "n_filters": [2,3], "ct_kernels": [(3,3),(5,5)], "batch_sizes": [16], "learning_rates": [1e-3],
               "patience": 10}


In [ ]:


rpss_train_list, rpss_val_list, rpss_test_list, predictions_list_nn, y_test_oh_list_nn = training.train_deepnet(xtrain_list, ytrain_list,
                                                                                                  xval_list,yval_list,
                                                                                                    xtest_list, ytest_list,
                                                                                                    training_type="tune", #train, tune or load
                                                                                                  architecture=architecture,
                                                                                                  architecture_params=architecture_params, #if train
                                                                                                  tuning_grid=tuning_grid, #if tune
                                                                                                  predictor="mean", #mean or stacked 
                                                                                                modname = model,
                                                                                                obs=obs, week=week,
                                                                                                epochs=100,
                                                                                                batch_size=16, #if not tuning
                                                                                                learning_rate=1e-3 #if not tuning
                                                                                                )

## Skill maps

In [ ]:
#make a mask based on the training data with less than 3 labels
def count_unique(values):
    return len(np.unique(values))
# Apply the function along the time dimension ('T')
y_test_terciled = y_test_oh_list_nn[0].argmax('category') 
unique_counts = xr.apply_ufunc(count_unique, y_test_terciled, input_core_dims=[['T']], vectorize=True)
# Mask grid points with less than 3 unique labels or NaNs
mask1 = (unique_counts <3)
mask2 = np.isnan(y).any(dim='T')
#combine masks
mask = mask1 | mask2

In [ ]:
cbar_kwargs = {'shrink': 0.7,'spacing': 'proportional'}

plots.plot_rpss_deepnet(rpss_train_list, rpss_val_list, rpss_test_list,model=model, obs=obs, week=week, architecture=architecture, mask=mask
                        , cbar_kwargs=cbar_kwargs, custom_title = None)

## Reliability Plot

In [ ]:
predictions_elr = [predictions_list_elr[i]for i in range(n_bootstraps)]
#stack predictions along bootstrap dimension
predictions_masked_elr = xr.concat(predictions_elr, dim='T')

y_test_oh_masked_elr = [y_test_oh_list_elr[i].where(~mask) for i in range(n_bootstraps)]
#stack predictions along bootstrap dimension
y_test_oh_masked_elr = xr.concat(y_test_oh_masked_elr, dim='T')


t_bn_elr = y_test_oh_masked_elr.sel(category='below').values
y_pred_bn_elr = predictions_masked_elr.sel(category='below').values

t_n_elr  = y_test_oh_masked_elr.sel(category='normal').values
y_pred_n_elr = predictions_masked_elr.sel(category='normal').values

t_an_elr = y_test_oh_masked_elr.sel(category='above').values
y_pred_an_elr = predictions_masked_elr.sel(category='above').values



In [ ]:
predictions_nn = [predictions_list_nn[i]for i in range(n_bootstraps)]
#stack predictions along bootstrap dimension
predictions_masked_nn = xr.concat(predictions_nn, dim='T')

y_test_oh_masked_nn = [y_test_oh_list_nn[i].where(~mask) for i in range(n_bootstraps)]
#stack predictions along bootstrap dimension
y_test_oh_masked_nn = xr.concat(y_test_oh_masked_nn, dim='T')


t_bn_nn = y_test_oh_masked_nn.sel(category='below').values
y_pred_bn_nn = predictions_masked_nn.sel(category='below').values

t_n_nn  = y_test_oh_masked_nn.sel(category='normal').values
y_pred_n_nn = predictions_masked_nn.sel(category='normal').values

t_an_nn = y_test_oh_masked_nn.sel(category='above').values
y_pred_an_nn = predictions_masked_nn.sel(category='above').values


In [ ]:
plots.reliability_diagram_compare(y_pred_bn_nn, t_bn_nn, y_pred_bn_elr, t_bn_elr, model=model, obs=obs, title=f'{week}-Below Normal')
plots.reliability_diagram_compare(y_pred_n_nn, t_n_nn, y_pred_n_elr, t_n_elr, model=model, obs=obs, title=f'{week}-Normal')
plots.reliability_diagram_compare(y_pred_an_nn, t_an_nn, y_pred_an_elr, t_an_elr, model=model, obs=obs, title=f'{week}-Above Normal')

In [ ]:
t_nn = y_test_oh_masked_nn.values
y_pred_nn = predictions_masked_nn.values

t_elr = y_test_oh_masked_elr.values
y_pred_elr = predictions_masked_elr.values


plots.reliability_diagram_compare(y_pred_nn, t_nn, y_pred_elr, t_elr, model=model, obs=obs, title=f'{week}-All')